In [ ]:
""" This notebook can be used to re-orientate and pad the HC images to the MS specs."""

In [ ]:
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from nibabel.orientations import axcodes2ornt, ornt_transform, apply_orientation

ms_root = Path('/media/yannik/Seagate Portable Drive/dfg_plexus/T1s/')
hc_root = Path('/media/yannik/Seagate Portable Drive/dfg_plexus/HC_T1s___freesurfer/')

ms_img = nib.load(ms_root / "Subject001_T1.nii")
ms_data = ms_img.get_fdata()
hc_img = nib.load(hc_root / "hc0001_T1.nii")
hc_data = hc_img.get_fdata()

In [ ]:
# Plotting the data
fig, ax = plt.subplots(2, 10, figsize=(12, 6))
for i, slice in enumerate(np.arange(0, 200, 20)):
    ax[0, i].imshow(ms_data[:, :, slice], cmap='gray')
    ax[1, i].imshow(hc_data[:, :, slice], cmap='gray')
plt.show()

In [ ]:
print(f"{ms_data.shape=}\t{ms_data.min()=}\t{ms_data.max()=}\t{ms_data.mean()=}")
print(f"{hc_data.shape=}\t{hc_data.min()=}\t{hc_data.max()=}\t{hc_data.mean()=}")

In [ ]:
ms_orientation = nib.aff2axcodes(ms_img.affine)
hc_orientation = nib.aff2axcodes(hc_img.affine)
print(f"{ms_orientation=}")
print(f"{hc_orientation=}")

In [ ]:
def normalize_and_pad_ct(input_path: Path,
                         output_dir: Path,
                         target_shape: tuple[int, int, int],
                         cohort_max: float | None = None,
                         normalization: str | None = None,
                         padding_mode: str = 'constant',
                         target_orientation: tuple[str, str, str] = ('R', 'A', 'S')):
    """
    Normalizes and pads a CT scan using cohort-specific max intensity.

    Args:
        input_path (Path): Path to input NIfTI file
        output_dir (Path): Directory to save normalized files
        target_shape (tuple): Desired output shape (e.g., (256,256,256))
        cohort_max (float): Precomputed maximum intensity for the cohort
        normalization (str): Normalization method ('minmax', 'zscore', 'cohort', None)
        padding_mode: 'constant' (zero-pad) or 'edge' (replicate last slice)
        target_orientation (tuple): Target axis codes (e.g., ('R','A','S') for RAS+)
    """
    # Load data
    img = nib.load(input_path)

    # --- Orientation Correction ---
    current_orientation = nib.aff2axcodes(img.affine)

    if current_orientation != target_orientation:

        current_ornt = axcodes2ornt(current_orientation)
        target_ornt = axcodes2ornt(target_orientation)
        transform = ornt_transform(current_ornt, target_ornt)

        reoriented_img = img.as_reoriented(transform)

    # --- Padding ---
    data = reoriented_img.get_fdata()
    pad_width = []
    for i, (current, target) in enumerate(zip(data.shape, target_shape)):
        pad_total = max(0, target - current)
        pad_before = pad_total // 2
        pad_after = pad_total - pad_before
        pad_width.append((pad_before, pad_after))

    padded_data = np.pad(
        data,
        pad_width,
        mode=padding_mode,
        constant_values=0
    )

    # --- Affine Adjustment ---
    affine = reoriented_img.affine.copy()
    try:
        for i in range(3):
            if pad_width[i][0] > 0:
                spacing = np.linalg.norm(affine[:3, i])
                if np.isfinite(spacing):
                    affine[:3, 3] -= pad_width[i][0] * spacing * affine[:3, i]
    except:
        pass

    # --- Normalization ---
    if normalization == 'minmax':
        normalized_data = (padded_data - padded_data.min()) / (padded_data.max() - padded_data.min())
    elif normalization == 'zscore':
        normalized_data = (padded_data - np.mean(padded_data)) / np.std(padded_data)
    elif normalization == 'cohort':
        assert cohort_max is not None, "Cohort max must be provided for 'cohort' normalization."
        normalized_data = (padded_data / cohort_max) * 255
    elif normalization is None:
        normalized_data = padded_data
    else:
        raise ValueError("Invalid normalization method. Choose 'minmax', 'zscore', or None.")

    final_data = np.clip(normalized_data, 0, 255)

    # --- Save ---
    output_path = output_dir / f"{Path(input_path).name}"
    header = nib.Nifti1Header()
    header.set_qform(affine, code='scanner')
    header.set_sform(affine, code='scanner')
    nib.save(nib.Nifti1Image(final_data, affine, header=header), output_path)

    return final_data, output_path

In [ ]:
# Convert all HC data
input_files = list(hc_root.glob('*.nii'))
target_dir = hc_root.parent / 'HC_T1s___freesurfer___reorientated_padded/'
Path(target_dir).mkdir(parents=True, exist_ok=True)

for file_path in tqdm(input_files, desc="Processing cohort"):
    normalize_and_pad_ct(
        input_path=Path(file_path),
        output_dir=target_dir,
        target_shape=(256, 256, 256),
        padding_mode='constant',  # or 'edge'
        target_orientation=('L', 'I', 'A')
    )

In [ ]:
# Visual comparison of original and preprocessed data
final_hc_data = normalize_and_pad_ct(
    input_path=hc_root / "hc0001_T1.nii",
    output_dir=target_dir,
    target_shape=(256, 256, 256),
    cohort_max=hc_max,
    padding_mode='constant',
    target_orientation=('L', 'I', 'A')
)[0]

fig, ax = plt.subplots(3, 10, figsize=(12, 6))
for i, slice in enumerate(np.arange(0, 200, 20)):
    ax[0, i].imshow(ms_data[:, :, slice], cmap='gray')
    ax[1, i].imshow(hc_data[:, :, slice], cmap='gray')
    ax[2, i].imshow(final_hc_data[:, :, slice], cmap='gray')
    for j in range(3):
        ax[j, i].set_xticks([])
        ax[j, i].set_yticks([])
        ax[j, i].set_title(f"Slice {slice}")
plt.show()

In [ ]:
print(f"{ms_data.shape=}\t{ms_data.min()=}\t{ms_data.max()=}\t{ms_data.mean()=}")
print(f"{hc_data.shape=}\t{final_hc_data.min()=}\t{final_hc_data.max()=}\t{final_hc_data.mean()=}")